# Sentiment Analysis on Amazon Reviews

In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk

In [3]:
nltk.download('vader_lexicon') # VADER Lexicon uses this dictionary to look up scoresfor each word

[nltk_data] Downloading package vader_lexicon to C:\Users\Hari
[nltk_data]     Prasanna M\AppData\Roaming\nltk_data...


True

## Sentiment Analysis using VADER (Valence Aware Dictionary and Sentiment Reasoner)

VADER is a rule-based approach for sentiment analysis. No training or preprocessing or feature extraction is required.

In [6]:
from nltk.sentiment import SentimentIntensityAnalyzer

In [10]:
review = "Good case, Excellent value." # Get a review from Amazon and paste it here.

sentiment = SentimentIntensityAnalyzer().polarity_scores(review)
print(sentiment)

{'neg': 0.0, 'neu': 0.1, 'pos': 0.9, 'compound': 0.8402}


As the compound score is **more than 0.05**, the sentiment is **positive** in this review.

In [12]:
review = "So there is no way for me to plug it in here in the US unless I go by a converter." 

sentiment = SentimentIntensityAnalyzer().polarity_scores(review)
print(sentiment)

{'neg': 0.12, 'neu': 0.88, 'pos': 0.0, 'compound': -0.3535}


In [13]:
review = """Project Hail Mary is the kind of movie 
that reminds you why you love cinema.""" 

sentiment = SentimentIntensityAnalyzer().polarity_scores(review)
print(sentiment)

{'neg': 0.0, 'neu': 0.703, 'pos': 0.297, 'compound': 0.6705}


## Machine Learning Based Approach: NLTK + Scikit-learn

Steps:

* Preprocessing
* Train-test Split
* Feature Extraction on Reviews using TF-IDF
* Model Training
* Evaluation

### Preprocessing

Preprocessing involves converting reviews into lowercase, removing punctuations, numbers, stop words, and then lemmatizing (converting the word to its root form).

In [14]:
nltk.download('stopwords') # Download stop words from NLTK package
nltk.download('wordnet') # Vocabulary for lemmatization
nltk.download('punkt_tab') # Tokenizer Rules

[nltk_data] Downloading package stopwords to C:\Users\Hari Prasanna
[nltk_data]     M\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to C:\Users\Hari Prasanna
[nltk_data]     M\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to C:\Users\Hari Prasanna
[nltk_data]     M\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [15]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [16]:
stop_words = stopwords.words('english')
stop_words

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [39]:
lemmatizer = WordNetLemmatizer() # Converting plural into singular form
lemmatizer.lemmatize('exciting', pos = 'v')

'excite'

In [55]:
import re

def preprocess(review):
    # Convert into lowercase
    lowercase_review = review.lower()

    # Remove punctuation and numbers
    punct_nums_removed = re.sub(r'[^a-z \s]', '', lowercase_review) 

    # Tokenize: convert sentence into list of words
    tokens = word_tokenize(punct_nums_removed)

    # Remove stop words
    stopwords_removed = [token for token in tokens if token not in stop_words]

    # Lemmaize
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in stopwords_removed]
    return " ".join(lemmatized_tokens)

In [56]:
sample_review = "The product WAS TOO BAD, one of the worst products ever! 0/10"

In [57]:
preprocess(sample_review)

'product bad one worst product ever'

In [59]:
reviews = "So there is no way for me to plug it in here in the US unless I go by a converter.	0"
features, target = reviews.split("\t")

In [60]:
features

'So there is no way for me to plug it in here in the US unless I go by a converter.'

In [61]:
target

'0'

In [64]:
cleaned_reviews = []
targets = []

with open('amazon_cells_labelled.txt', 'r') as file:
    for review in file:
        review, target = review.strip().split("\t")
        cleaned_review = preprocess(review)
        cleaned_reviews.append(cleaned_review)
        targets.append(target)

In [67]:
cleaned_reviews[:5]

['way plug u unless go converter',
 'good case excellent value',
 'great jawbone',
 'tied charger conversation lasting minutesmajor problem',
 'mic great']

In [68]:
targets[:5]

['0', '1', '1', '0', '1']

In [71]:
targets.count("1") # 500 positive reviews

500

In [72]:
targets.count("0") # 500 negative review

500

### Train-test Split

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(cleaned_reviews,
                                                    targets,
                                                    test_size=0.2,
                                                    stratify=targets)

In [74]:
len(X_train) == len(y_train)

True

In [75]:
len(X_test) == len(y_test)

True

### Feature Extraction

In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [77]:
vectorizer = TfidfVectorizer()

In [78]:
X_train_features = vectorizer.fit_transform(X_train)

In [84]:
X_train_features.shape # 800 reviews, each with 1385 features

(800, 1385)

In [85]:
vectorizer.get_feature_names_out() # Features extracted from the reviews

array(['abhor', 'ability', 'able', ..., 'youd', 'youll', 'zero'],
      dtype=object)

### Model Training

In [87]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [89]:
logistic = LogisticRegression()

In [90]:
logistic.fit(X_train_features, y_train)

LogisticRegression()

#### Train performance

In [91]:
y_pred_train = logistic.predict(X_train_features)

In [92]:
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score

balanced_accuracy_train = balanced_accuracy_score(y_train, y_pred_train)
precision_train = precision_score(y_train, y_pred_train, pos_label='1')
recall_train = recall_score(y_train, y_pred_train, pos_label='1')

In [93]:
print("Train Performance")

print("Accuracy:",balanced_accuracy_train)
print("Precision:",precision_train)
print("Recall:",recall_train)

Train Performance
Accuracy: 0.95625
Precision: 0.9715762273901809
Recall: 0.94


#### Test Performance

In [105]:
y_pred = logistic.predict(X_test_features)

In [112]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

accuracy_test = accuracy_score(y_test, y_pred)
precision_test = precision_score(y_test, y_pred, pos_label='1')  
recall_test = recall_score(y_test, y_pred, pos_label='1')        

print("Accuracy:", accuracy_test)
print("Precision:", precision_test)
print("Recall:", recall_test)


Accuracy: 0.79
Precision: 0.8536585365853658
Recall: 0.7


#### Using Naive Bayes Model

In [95]:
X_test_features = vectorizer.transform(X_test)
X_test_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 831 stored elements and shape (200, 1385)>

In [99]:
from sklearn.naive_bayes import MultinomialNB

multinomial_nb = MultinomialNB()
multinomial_nb.fit(X_train_features, y_train)

MultinomialNB()

In [113]:
from sklearn.metrics import accuracy_score

# Predict labels for the test set
y_pred_nb = multinomial_nb.predict(X_test_features)

# Calculate accuracy
accuracy_test_nb = accuracy_score(y_test, y_pred_nb)
precision_test = precision_score(y_test, y_pred_nb, pos_label='1')  
recall_test = recall_score(y_test, y_pred_nb, pos_label='1')        

print("Accuracy:", accuracy_test)
print("Precision:", precision_test)
print("Recall:", recall_test)


Accuracy: 0.79
Precision: 0.8058252427184466
Recall: 0.83
